# IST DBM 2025 Project - Omarion Aubert, Franciszek Dobrowolski
## Domain

We've decided to design a simple data base to answer questions related to trends in gastronomy:

## Questions in natural language

1. What is the favorite dish for males?
2. Which city has the most seniors eating?
3. What’s the most popular franchise in Louisiana, LA?
4. For every customer, find a restaurant in which they spend the most money in one day.
5. Find the restaurants, in which customers and employees have a different favourite dish.
6. List the restaurants, which had the most customers per month last year.

## ER diagram and relational schema

<img src="diagrams/ER_diagram_Aubert_Dobrowolski.png">

<img src="diagrams/Realtional_schema_Aubert_Dobrowolski.png">

## Connecting to Postgres, creating a data base and connecting to it

In [25]:
#installing libraries if necessary
! pip install psycopg2
! pip install pandas

import psycopg2
import pandas as pd
import pandas.io.sql as psql

from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

# Connecting to the postgreSQL
try:
    con = psycopg2.connect(user = "postgres",
                                  password = "postgres",
                                  host = "127.0.0.1",
                                  port = "5432")
    
    con.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT);
    print("Connected Successfully to PostgreSQL server!!")
    
    cursor = con.cursor();
except (Exception, psycopg2.Error) as error :
     print ("Error while connecting to PostgreSQL", error)

Connected Successfully to PostgreSQL server!!


In [26]:
# Creating a DB
name_Database   = "gastronomy";
sqlCreateDatabase = "CREATE DATABASE "+name_Database+";"

try:
    cursor.execute(sqlCreateDatabase);
    print("Database '"+name_Database+"' Created Successfully!")
except (Exception, psycopg2.Error) as error :
    print("Error While Creating the DB: ",error)

# get a new connection for the new DB.
con = psycopg2.connect(user = "postgres",
                       password = "postgres",
                       host = "127.0.0.1",
                       port = "5432",
                       database = "gastronomy")

try:
    cursor = con.cursor();
    print("connected again to the server and cusor now on Gastronomy DB !!")
except (Exception, psycopg2.Error) as error:
    print("Error in Connection",error)



Database 'gastronomy' Created Successfully!
connected again to the server and cusor now on Gastronomy DB !!


## Creating the tables

In [27]:
#creating customer table
try:
    customerTable="customer"
    create_customerTablee_query = '''CREATE TABLE '''+ customerTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        name TEXT NOT NULL,
        age INT NOT NULL,
        gender TEXT
    );'''
    cursor.execute(create_customerTablee_query)
    con.commit()
    print("Table ("+ customerTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

#creating city table
try:
    cityTable="city"
    create_cityTable_query = '''CREATE TABLE '''+ cityTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        name TEXT NOT NULL,
        state TEXT NOT NULL
    ); '''
    cursor.execute(create_cityTable_query)
    con.commit()
    print("Table ("+ cityTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

#creating restaurant table
try:
    restaurantTable="restaurant"
    create_restaurantTable_query = '''CREATE TABLE '''+ restaurantTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        name TEXT NOT NULL,
        franchise TEXT,
        city_id INT NOT NULL,
        FOREIGN KEY (city_id) REFERENCES city(id)
    ); '''
    cursor.execute(create_restaurantTable_query)
    con.commit()
    print("Table ("+ restaurantTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

#creating food table
try:
    foodTable="food"
    create_foodTable_query = '''CREATE TABLE '''+ foodTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        name TEXT NOT NULL,
        price INT NOT NULL
    ); '''
    cursor.execute(create_foodTable_query)
    con.commit()
    print("Table ("+ foodTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

#creating employee table
try:
    employeeTable="employee"
    create_employeeTable_query = '''CREATE TABLE '''+ employeeTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        name TEXT NOT NULL,
        restaurant_id INT NOT NULL,
        favourite_food_id INT NOT NULL,
        FOREIGN KEY (restaurant_id) REFERENCES restaurant(id),
        FOREIGN KEY (favourite_food_id) REFERENCES food(id)
    ); '''
    cursor.execute(create_employeeTable_query)
    con.commit()
    print("Table ("+ employeeTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

#creating orders table
try:
    orderTable="orders"
    create_orderTable_query = '''CREATE TABLE '''+ orderTable+''' 
    (
        id INT PRIMARY KEY NOT NULL,
        order_date DATE NOT NULL,
        customer_id INT NOT NULL,
        restaurant_id INT NOT NULL,
        food_id INT NOT NULL,
        FOREIGN KEY (customer_id) REFERENCES customer(id),
        FOREIGN KEY (restaurant_id) REFERENCES restaurant(id),
        FOREIGN KEY (food_id) REFERENCES food(id)
    ); '''
    cursor.execute(create_orderTable_query)
    con.commit()
    print("Table ("+ orderTable +") created successfully in PostgreSQL ")
except (Exception, psycopg2.Error) as error:
    con.rollback()
    print("Error While Creating the DB: ",error)

Table (customer) created successfully in PostgreSQL 
Table (city) created successfully in PostgreSQL 
Table (restaurant) created successfully in PostgreSQL 
Table (food) created successfully in PostgreSQL 
Table (employee) created successfully in PostgreSQL 
Table (orders) created successfully in PostgreSQL 


## Filling the tables
The data generated using Mockaroo.

In [28]:
import csv
from psycopg2 import sql

def load_csv_to_table(cursor, table_name, csv_path):
    with open(csv_path, 'r', encoding='utf-8') as f:
        next(f)  # pominięcie nagłówka
        cursor.copy_expert(
            sql.SQL("COPY {} FROM STDIN WITH CSV").format(sql.Identifier(table_name)),
            f
        )

try:
    load_csv_to_table(cursor, "customer", "data/customers.csv")
    con.commit()
    print("customer loaded")
    
    load_csv_to_table(cursor, "city", "data/cities.csv")
    con.commit()
    print("city loaded")
    
    load_csv_to_table(cursor, "restaurant", "data/restaurants.csv")
    con.commit()
    print("restaurant loaded")
    
    load_csv_to_table(cursor, "food", "data/foods.csv")
    con.commit()
    print("food loaded")
    
    load_csv_to_table(cursor, "employee", "data/employees.csv")
    con.commit()
    print("employee loaded")
    
    load_csv_to_table(cursor, "orders", "data/orders.csv")
    con.commit()
    print("orders loaded")

except Exception as e:
    con.rollback()
    print("Error:", e)


customer loaded
city loaded
restaurant loaded
food loaded
employee loaded
orders loaded


## Queries

### Query nr 1
What is the favorite dish for men?

π{f.name} ( σ{c.gender='male'} ( (order ⋈{o.customer_id=c.id} customer) ⋈{o.food_id=f.id} food ) )


In [29]:
query1 = psql.read_sql("""
    SELECT f.name AS favorite_dish, COUNT(*) AS orders_count
    FROM orders o
    JOIN customer c ON o.customer_id = c.id
    JOIN food f ON o.food_id = f.id
    WHERE c.gender = 'male'
    GROUP BY f.name
    ORDER BY orders_count DESC
    LIMIT 3;
""", con)
display(query1)

/tmp/ipykernel_56227/2667868869.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query1 = psql.read_sql("""


,favorite_dish,orders_count
0,Chicken Tikka Masala,17
1,Pad Thai,15
2,Pizza,14


### Query nr 2
Which city has the most seniors eating?

π{ci.name, o.customer_id} ( σ{c.age >= 65} ( (((order ⋈{o.customer_id=c.id} customer) ⋈{o.restaurant_id=r.id} restaurant) ⋈{r.city_id=ci.id} city) ) )


In [30]:
query2 = psql.read_sql("""
    SELECT ci.name AS city_name, COUNT(DISTINCT o.customer_id) AS senior_count
    FROM orders o
    JOIN customer c ON o.customer_id = c.id
    JOIN restaurant r ON o.restaurant_id = r.id
    JOIN city ci ON r.city_id = ci.id
    WHERE c.age >= 65
    GROUP BY ci.name
    ORDER BY senior_count DESC
    LIMIT 3;
""", con)
display(query2)

/tmp/ipykernel_56227/1052275923.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query2 = psql.read_sql("""


,city_name,senior_count
0,Baton Rouge,14
1,Albany,13
2,Moreno Valley,12


### Query nr 3
What’s the most popular franchise in Louisiana, LA?

π{r.franchise, o.id} ( σ{c.state = 'LA'} ( ( (order ⋈{o.restaurant_id = r.id} restaurant ) ⋈{r.city_id = c.id} city ) ) )

In [31]:
query3 = psql.read_sql("""
    SELECT r.franchise, COUNT(*) AS orders_count
    FROM orders o
    JOIN restaurant r ON o.restaurant_id = r.id
    JOIN city c ON r.city_id = c.id
    WHERE c.state = 'LA'
    GROUP BY r.franchise
    ORDER BY orders_count DESC
    LIMIT 3;
""", con)
display(query3)

/tmp/ipykernel_56227/1199489095.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query3 = psql.read_sql("""


,franchise,orders_count
0,"Flamingo, chilean",12
1,Ostrich,10
2,Common duiker,9


### Query nr 4
For every customer, find a restaurant in which they spend the most money in one day.

Simplified, because of sum, groups, etc.

R = (order ⋈{o.customer_id = c.id} customer)
      ⋈{o.food_id = f.id} food
      ⋈{o.restaurant_id = r.id} restaurant

Rfinal = π{c.id, c.name, o.restaurant_id, r.name, o.order_date}(R)


In [32]:
query4 = psql.read_sql("""
    SELECT
        c.id AS customer_id,
        c.name AS customer_name,
        o.restaurant_id,
        r.name AS restaurant_name,
        o.order_date,
        SUM(f.price) AS total_spent
    FROM
        orders o
    JOIN customer c ON o.customer_id = c.id
    JOIN food f ON o.food_id = f.id
    JOIN restaurant r ON o.restaurant_id = r.id
    GROUP BY c.id, c.name, o.restaurant_id, r.name, o.order_date
    HAVING SUM(f.price) = (
        SELECT MAX(total)
        FROM (
            SELECT SUM(f2.price) AS total
            FROM orders o2
            JOIN food f2 ON o2.food_id = f2.id
            WHERE o2.customer_id = c.id
            GROUP BY o2.restaurant_id, o2.order_date
        ) AS sub
    );
""", con)
display(query4)

/tmp/ipykernel_56227/1176186065.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query4 = psql.read_sql("""


,customer_id,customer_name,restaurant_id,restaurant_name,order_date,total_spent
0,289,Dunridge,1,Weber-Rippin,2024-07-07,33
1,291,Gehricke,20,Koelpin-Rowe,2024-07-27,49
2,18,Drinkel,61,Lockman Group,2024-04-19,22
3,45,Phin,37,Paucek-Schulist,2024-03-24,45
4,11,Getcliff,22,Halvorson-Shanahan,2024-10-23,19
...,...,...,...,...,...,...
275,6,Collcott,25,Zemlak Group,2024-12-28,32
276,190,de Verson,35,Cassin-Gulgowski,2024-01-29,41
277,195,Pryor,33,Keebler-Ullrich,2024-07-21,39
278,201,Kubu,57,Thompson-Roob,2024-04-08,40


### Query nr 5
Find the restaurants, in which customers and employees have a different favourite dish.

Simplified:

CustOrders = order ⋈{o.restaurant_id = r.id} restaurant

EmpFav = employee ⋈{e.restaurant_id = r.id} restaurant


Rdiff = σ{o.food_id != e.favourite_food_id}(CustOrders ⋈{r.id = e.restaurant_id} EmpFav)


Rfinal = π{r.id, r.name}(Rdiff)

In [33]:
query5 = psql.read_sql("""
    WITH customer_fav AS (
    SELECT
        o.restaurant_id,
        o.food_id,
        COUNT(*) AS cnt
    FROM orders o
    GROUP BY o.restaurant_id, o.food_id
),
customer_top AS (
    SELECT restaurant_id, food_id
    FROM customer_fav cf
    WHERE cf.cnt = (
        SELECT MAX(cf2.cnt)
        FROM customer_fav cf2
        WHERE cf2.restaurant_id = cf.restaurant_id
    )
),
employee_fav AS (
    SELECT
        e.restaurant_id,
        o.food_id,
        COUNT(*) AS cnt
    FROM employee e
    JOIN orders o ON e.restaurant_id = o.restaurant_id
    GROUP BY e.restaurant_id, o.food_id
),
employee_top AS (
    SELECT restaurant_id, food_id
    FROM employee_fav ef
    WHERE ef.cnt = (
        SELECT MAX(ef2.cnt)
        FROM employee_fav ef2
        WHERE ef2.restaurant_id = ef.restaurant_id
    )
)
SELECT DISTINCT r.id, r.name
FROM restaurant r
JOIN customer_top ct ON r.id = ct.restaurant_id
JOIN employee_top et ON r.id = et.restaurant_id
WHERE ct.food_id <> et.food_id;

""", con)
display(query5)

/tmp/ipykernel_56227/1810977373.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query5 = psql.read_sql("""


,id,name
0,1,Weber-Rippin
1,4,Kassulke Group
2,6,Raynor-Dibbert
3,7,Collins-Reinger
4,8,Hahn-Ruecker
5,14,Herman Group
6,16,"Hayes, Grant and Blick"
7,18,"Mann, Langosh and Legros"
8,22,Halvorson-Shanahan
9,23,"Shanahan, Crist and Ondricka"


### Query nr 6
List the restaurants, which had the most customers per month last year.

R1 = σ{YEAR(o.order_date) = YEAR(CURRENT_DATE)-1}(order)

R2 = R1 ⋈{o.restaurant_id = r.id} restaurant

Rfinal = π{r.name, DATE_TRUNC('month', o.order_date), o.customer_id}(R2)

In [34]:
query6 = psql.read_sql("""
    SELECT r.name,
       DATE_TRUNC('month', o.order_date) AS month,
       COUNT(DISTINCT o.customer_id) AS customers_count
    FROM orders o
    JOIN restaurant r ON o.restaurant_id = r.id
    WHERE EXTRACT(YEAR FROM o.order_date) = EXTRACT(YEAR FROM CURRENT_DATE) - 1
    GROUP BY r.name, DATE_TRUNC('month', o.order_date)
    ORDER BY customers_count DESC
    LIMIT 3;
""", con)
display(query6)

/tmp/ipykernel_56227/4032703648.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  query6 = psql.read_sql("""


,name,month,customers_count
0,Schultz Group,2024-04-01 00:00:00+00:00,5
1,Koelpin-Rowe,2024-07-01 00:00:00+00:00,4
2,"Hayes, Prosacco and Littel",2024-05-01 00:00:00+00:00,4
